In [19]:
import pandas as pd, sqlite3


In [25]:
def extract(path):
    df = pd.read_csv(path)
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('columns that arrived as text:',
          [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])])
    print('nulls per column:')
    print(df.isnull().sum()[lambda s: s > 0].to_string())
    print('exact duplicates:', df.duplicated().sum())
    return df

In [37]:
distribution_raw = extract('../data/distribution_centers.csv')

def transform_distribution(df):
    out = df.drop_duplicates().copy()

    out['surge_capacity'] = (out['surge_capacity'].astype(str)
                         .str.replace('~', '', regex=False)
                         .str.strip().astype(int))

    return out

distribution_typed = transform_distribution(distribution_raw)
print('rows:', len(distribution_raw), '->', len(distribution_typed))

extracted 5 rows, 8 columns
columns that arrived as text: ['dc_id', 'dc_name', 'city', 'state', 'surge_capacity']
nulls per column:
cost_per_shipment    1
exact duplicates: 0
rows: 5 -> 5


In [ ]:
product_raw = extract('../data/product_catalog.csv')
def transform_product(df):
    out = df.drop_duplicates().copy()

    out['']
product_raw.head()

extracted 45 rows, 7 columns
columns that arrived as text: ['sku', 'product_name', 'category', 'subcategory', 'supplier', 'package_size']
nulls per column:
subcategory    13
unit_cost       5
exact duplicates: 0


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


In [23]:
store_raw = extract('../data/store_locations.csv')
store_raw.head()

extracted 14 rows, 8 columns
columns that arrived as text: ['store_id', 'store_name', 'city', 'state', 'region']
nulls per column:
latitude     1
longitude    1
exact duplicates: 0


,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


In [42]:
transactions_raw = extract('../data/walmart_transactions.csv')
def transform_transactions(df):
    out = df.drop_duplicates().copy()

    out['unit_price'] = (out['unit_price'].astype(str)
                         .str.replace('$', '', regex=False)
                         .str.strip().astype(float))

    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    out['quantity'] = out['quantity'].astype(int)

    out['store_id'] = (out['store_id'].str.upper().str.strip()
                       .str.replace(r'[^A-Z0-9]', '', regex=True)
                       .apply(lambda s: f'{s[:2]}-{s[2:]}'))

    out['category'] = out['category'].str.strip().str.title()
    
    out['timestamp'] = pd.to_datetime(out['timestamp'],
                                      errors='coerce', format='mixed')
    return out
transactions_typed = transform_transactions(transactions_raw)
print('rows:', len(transactions_raw), '->', len(transactions_typed))
transactions_typed.head()

extracted 25562 rows, 8 columns
columns that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls per column:
timestamp    886
exact duplicates: 30
rows: 25562 -> 25532


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-13 09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,FL-412,2024-09-02 20:30:20,CNSOUP-01,CANNED SOUP,Grocery,1,2.02
2,TXN003338,FL-105,2024-09-06 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-30 06:29:08,ROKU-STD,Roku Streaming Stick,Electronics,1,28.59
4,TXN015797,FL-302,2024-09-15 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82
